In [2]:
!pip install numpy pandas matplotlib seaborn scikit-learn datasets

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 39.9 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 41.5 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 34.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 41.3 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 18.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 24.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.4/563.4 kB 5.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 21.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 37.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 75.5 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/

In [6]:
# --- Clustering task for testing embeddings ---

# we want to test 6 different embeddings on how they can determine clusters
# the gold standard is the category_gold

# we will use the KMeans clustering algorithm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.cluster import v_measure_score
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

from datasets import Dataset
import logging

# Configure logging
logging.basicConfig(
    filename='../logs/clustering_report.txt',           # Output file
    filemode='w',                    # 'w' to overwrite each run; use 'a' to append
    format='%(asctime)s - %(levelname)s - %(message)s',  # Log format
    level=logging.INFO,              # Minimum level: DEBUG, INFO, WARNING, ERROR, CRITICAL
    force=True,                # Force logging even if already configured
)

In [7]:
# Load gold sample

sample_gold = pd.read_csv('../data/test_task/subset_final_gold_sample.csv', sep=',')
sample_gold.head()

# Load embeddings of sample

path_root = '../data/test_task/pooled/'

merged_all = Dataset.load_from_disk(f'{path_root}merged_all')
merged_all = merged_all.to_pandas()

# Merge gold labels with embeddings

embs_gold_orig = sample_gold.merge(merged_all[['article_id', 'e-5', 'jina', 'memo', 'old', 'bge', 'gemma']], on='article_id')
embs_gold_orig.columns

Index(['index', 'text', 'category_gold', 'date', 'article_id', 'newspaper',
       'e-5', 'jina', 'memo', 'old', 'bge', 'gemma'],
      dtype='object')

In [10]:
# Suppose your embedding columns are named like this:
embeddings_cols = ["e-5", "jina", "memo", "old", "bge", "gemma"]

save_dict = {}

for col in embeddings_cols:
    logging.info("----------------------")
    logging.info(f"Evaluating embeddings from column: {col}")

    # filter out faulty rows
    df_valid = embs_gold_orig[
        embs_gold_orig[col].apply(lambda x: isinstance(x, np.ndarray) and not np.isnan(x).any())
    ]

    df_valid = df_valid[df_valid["category_gold"] != "Paratext"]

    # Prepare X and y
    X = np.vstack(df_valid[col].values)
    y = df_valid["category_gold"].values

    # number of clusters = number of gold categories
    n_clusters = np.unique(y).shape[0]

    # run kmeans
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X)

    # metrics
    ari = round(adjusted_rand_score(y, clusters), 3)
    v_score = round(v_measure_score(y, clusters), 3)

    print(f"{col} | n_clusters: {n_clusters} | ARI: {ari} | V-score: {v_score}")
    logging.info(f"{col} | ARI: {ari} | V-score: {v_score}")

    # save
    save_dict[col] = {
        "ari": ari,
        "v_score": v_score,
        "n_clusters": n_clusters
    }

# write results to file
with open("../logs/clustering_results_gold_3cats.txt", "a") as f:
    f.write("\n\nClustering report:\n")
    for col, metrics in save_dict.items():
        f.write(f"{col}: {metrics}\n")
    f.write("\n")

e-5 | n_clusters: 3 | ARI: 0.422 | V-score: 0.43
jina | n_clusters: 3 | ARI: 0.445 | V-score: 0.47
memo | n_clusters: 3 | ARI: 0.196 | V-score: 0.204
old | n_clusters: 3 | ARI: 0.554 | V-score: 0.568
bge | n_clusters: 3 | ARI: 0.345 | V-score: 0.395
gemma | n_clusters: 3 | ARI: 0.015 | V-score: 0.013
